# PaySim vs 无标签交易数据：相似性检验（Fusion）

> 目标：在无欺诈标签场景下，构建“可迁移性证据链”。本Notebook实现 Fusion.md 中的 **相似性检验**：

- **表格层（交易级）**：对齐金额分布（PaySim `amount` vs 无标签 `payment_amount`）
- **时序层（行为节奏）**：对齐到“交易间隔分布”（PaySim `Δstep` vs 无标签 `Δt` 秒）
- **图结构层（最关键）**：对齐账户交易图的结构统计（度分布、连通分量、重复边、reciprocity）

> 注：无标签数据字段来自 `Graph/graph_main/README.md`，其中账户字段默认使用 `debit_account_masked` 与 `bene_account_masked`；时间字段优先使用 `txn_dt`，缺失时回退 `tds_dt`。

In [ ]:
# ============ 配置区（按需修改） ============
from pathlib import Path

# PaySim 数据（已在工作区根目录）
PAYSIM_PATH = Path(r"Paysim.csv")

# 无标签数据：请改成你的真实交易CSV路径（默认先留空/示例）
UNLABELED_PATH = Path(r"xxx.csv")

# 无标签数据字段名（如果你的CSV列名不同，在这里改）
UNLABELED_COLS = {
    "amount": "payment_amount",
    "src": "debit_account_masked",
    "dst": "bene_account_masked",
    "txn_time": "txn_dt",
    "fallback_time": "tds_dt",
}

# 统计与图表参数
RANDOM_SEED = 42
MAX_ROWS = 0  # 0表示全量；>0时只读取前N行（调试用）
TOP_Q = [0.50, 0.90, 0.95, 0.99, 0.999]
HIST_BINS = 200


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import wasserstein_distance

sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 4)


In [ ]:
def _safe_read_csv(path: Path, nrows: int = 0) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"找不到文件: {path}")
    return pd.read_csv(path, nrows=None if nrows == 0 else nrows)

def _to_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _ensure_columns(df: pd.DataFrame, cols: list[str], df_name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{df_name} 缺少列: {missing}. 现有列示例: {list(df.columns)[:30]}")

def _qcurve(arr: np.ndarray, qs: list[float]) -> pd.DataFrame:
    arr = arr[~np.isnan(arr)]
    return pd.DataFrame({"q": qs, "value": [np.quantile(arr, q) for q in qs]})

def _tail_share(arr: np.ndarray, q: float) -> float:
    """Top-q_tail(如0.99)占比：大于等于该分位数的总和 / 总和"""
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan
    thr = np.quantile(arr, q)
    denom = arr.sum()
    if denom == 0:
        return np.nan
    return float(arr[arr >= thr].sum() / denom)


## 1) 读取数据 & 字段对齐

本节会：
- 读取 `Paysim.csv`（字段固定为：`step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud`）
- 读取无标签CSV，并用 `UNLABELED_COLS` 映射找到金额/账户/时间列
- 做基础清洗：金额转数值、去除 NaN/Inf

In [ ]:
# 读取 PaySim
paysim = _safe_read_csv(PAYSIM_PATH, nrows=MAX_ROWS).copy()

expected_paysim_cols = [
    "step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig",
    "nameDest","oldbalanceDest","newbalanceDest","isFraud"
 ]
_ensure_columns(paysim, expected_paysim_cols, "PaySim")

# 读取无标签数据
unlabeled = _safe_read_csv(UNLABELED_PATH, nrows=MAX_ROWS).copy()
_ensure_columns(unlabeled, list(UNLABELED_COLS.values()), "Unlabeled")

# 统一取出对齐字段
paysim_amount = _to_numeric_series(paysim["amount"]).replace([np.inf, -np.inf], np.nan)
unlabeled_amount = _to_numeric_series(unlabeled[UNLABELED_COLS["amount"]]).replace([np.inf, -np.inf], np.nan)

print("PaySim shape:", paysim.shape)
print("Unlabeled shape:", unlabeled.shape)
print("PaySim amount valid:", int(paysim_amount.notna().sum()), "/", len(paysim_amount))
print("Unlabeled amount valid:", int(unlabeled_amount.notna().sum()), "/", len(unlabeled_amount))


## 2) 表格层（交易级）：金额分布相似性

对齐字段：
- PaySim：`amount`
- 无标签：`payment_amount`（可在 `UNLABELED_COLS['amount']` 修改）

输出：
- 原始金额与 `log1p(amount)` 的直方图/核密度对比
- KS 距离、Wasserstein 距离
- 分位数曲线对比（Quantile Curve）
- 尾部占比：Top 1% / Top 0.1% 金额贡献

In [ ]:
# 清洗并转numpy
pa = paysim_amount.dropna().to_numpy(dtype=float)
ua = unlabeled_amount.dropna().to_numpy(dtype=float)

# 基础指标
amt_stats = pd.DataFrame({
    "dataset": ["PaySim", "Unlabeled"],
    "n": [len(pa), len(ua)],
    "mean": [pa.mean() if len(pa) else np.nan, ua.mean() if len(ua) else np.nan],
    "median": [np.median(pa) if len(pa) else np.nan, np.median(ua) if len(ua) else np.nan],
    "max": [pa.max() if len(pa) else np.nan, ua.max() if len(ua) else np.nan],
    "min": [pa.min() if len(pa) else np.nan, ua.min() if len(ua) else np.nan],
})
amt_stats

In [ ]:
# KS / Wasserstein（对原始amount；建议同时看log1p后更稳定）
ks_raw = stats.ks_2samp(pa, ua)
wd_raw = wasserstein_distance(pa, ua)

ks_log = stats.ks_2samp(np.log1p(pa), np.log1p(ua))
wd_log = wasserstein_distance(np.log1p(pa), np.log1p(ua))

dist_tbl = pd.DataFrame([
    {"space": "raw", "ks_stat": ks_raw.statistic, "ks_pvalue": ks_raw.pvalue, "wasserstein": wd_raw},
    {"space": "log1p", "ks_stat": ks_log.statistic, "ks_pvalue": ks_log.pvalue, "wasserstein": wd_log},
])
dist_tbl

In [ ]:
# 可视化：log1p金额分布对比
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(np.log1p(pa), bins=HIST_BINS, stat="density", kde=True, ax=axes[0], color="#4c78a8", label="PaySim")
sns.histplot(np.log1p(ua), bins=HIST_BINS, stat="density", kde=True, ax=axes[0], color="#f58518", label="Unlabeled")
axes[0].set_title("log1p(amount) 分布")
axes[0].legend()

# 分位数曲线
pq = _qcurve(pa, TOP_Q).rename(columns={"value": "PaySim"})
uq = _qcurve(ua, TOP_Q).rename(columns={"value": "Unlabeled"})
qdf = pq.merge(uq, on="q", how="outer").sort_values("q")
axes[1].plot(qdf["q"], qdf["PaySim"], marker="o", label="PaySim")
axes[1].plot(qdf["q"], qdf["Unlabeled"], marker="o", label="Unlabeled")
axes[1].set_xscale("linear")
axes[1].set_yscale("log")
axes[1].set_title("分位数曲线（y为log刻度）")
axes[1].set_xlabel("quantile")
axes[1].set_ylabel("amount")
axes[1].legend()

plt.tight_layout()
plt.show()

qdf

In [ ]:
# 尾部占比：Top 1% / Top 0.1% 金额贡献
tail_tbl = pd.DataFrame([
    {"dataset": "PaySim", "top1%_share": _tail_share(pa, 0.99), "top0.1%_share": _tail_share(pa, 0.999)},
    {"dataset": "Unlabeled", "top1%_share": _tail_share(ua, 0.99), "top0.1%_share": _tail_share(ua, 0.999)},
])
tail_tbl

## 3) 时序层（行为节奏）：交易间隔分布相似性

思路（Fusion.md）：不要硬对齐日历变量，而是对齐节奏统计。
- PaySim：对每个账户（`nameOrig`）按 `step` 排序，算相邻交易的 `Δstep`（单位：小时）
- 无标签：对每个账户（`debit_account_masked`）按 `txn_dt`（失败则用 `tds_dt`）排序，算相邻 `Δt`（单位：小时）

输出：
- `Δstep` vs `Δt` 的分布对比（log刻度）
- KS / Wasserstein（在 log 空间会更稳）

> 注意：两边现在都使用小时作为时间单位，可以直接比较数值大小和形态相似性。

In [ ]:
def _compute_interarrival_paysim(df: pd.DataFrame) -> np.ndarray:
    # 对 nameOrig 的相邻交易 step 差（单位：小时）
    tmp = df[["nameOrig", "step"]].copy()
    tmp["step"] = pd.to_numeric(tmp["step"], errors="coerce")
    tmp = tmp.dropna(subset=["nameOrig", "step"])
    tmp = tmp.sort_values(["nameOrig", "step"])
    dt = tmp.groupby("nameOrig")["step"].diff()
    dt = dt.dropna()
    dt = dt[(dt >= 0) & np.isfinite(dt)]
    return dt.to_numpy(dtype=float)

def _parse_time_series(s: pd.Series) -> pd.Series:
    # 尽量宽松解析
    ts = pd.to_datetime(s, errors="coerce")
    return ts

def _compute_interarrival_unlabeled(df: pd.DataFrame, src_col: str, time_col: str, fallback_col: str | None = None) -> np.ndarray:
    # 对 src_col 的相邻交易时间差（单位：小时）
    tmp = df[[src_col, time_col]].copy()
    tmp[time_col] = _parse_time_series(tmp[time_col])
    if tmp[time_col].isna().all() and fallback_col is not None:
        tmp = df[[src_col, fallback_col]].copy()
        tmp[fallback_col] = _parse_time_series(tmp[fallback_col])
        tmp = tmp.rename(columns={fallback_col: time_col})
    tmp = tmp.dropna(subset=[src_col, time_col])
    tmp = tmp.sort_values([src_col, time_col])
    # 转换为小时：total_seconds() / 3600
    dt = tmp.groupby(src_col)[time_col].diff().dt.total_seconds() / 3600
    dt = dt.dropna()
    dt = dt[(dt >= 0) & np.isfinite(dt)]
    return dt.to_numpy(dtype=float)

p_dt = _compute_interarrival_paysim(paysim)
u_dt = _compute_interarrival_unlabeled(
    unlabeled,
    src_col=UNLABELED_COLS["src"],
    time_col=UNLABELED_COLS["txn_time"],
    fallback_col=UNLABELED_COLS.get("fallback_time"),
 )

print("PaySim Δstep (hours) n:", len(p_dt))
print("Unlabeled Δt (hours) n:", len(u_dt))

In [ ]:
# 在log空间对比形态（避免长尾影响）
p_log = np.log1p(p_dt)
u_log = np.log1p(u_dt)

ks_dt = stats.ks_2samp(p_log, u_log) if (len(p_log) and len(u_log)) else None
wd_dt = wasserstein_distance(p_log, u_log) if (len(p_log) and len(u_log)) else np.nan

pd.DataFrame([{
    "space": "log1p(Δ)",
    "ks_stat": np.nan if ks_dt is None else ks_dt.statistic,
    "ks_pvalue": np.nan if ks_dt is None else ks_dt.pvalue,
    "wasserstein": wd_dt,
}])

In [ ]:
# 分布图（log空间）
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
sns.histplot(p_log, bins=HIST_BINS, stat="density", kde=True, ax=ax, color="#4c78a8", label="PaySim log1p(Δstep hours)")
sns.histplot(u_log, bins=HIST_BINS, stat="density", kde=True, ax=ax, color="#f58518", label="Unlabeled log1p(Δt hours)")
ax.set_title("交易间隔分布（log1p空间，单位：小时）")
ax.set_xlabel("log1p(Δ hours)")
ax.legend()
plt.tight_layout()
plt.show()

## 4) 图结构层（最关键）：账户交易图相似性

对齐字段：
- PaySim：`nameOrig` → `nameDest`（有向边）
- 无标签：`debit_account_masked` → `bene_account_masked`（有向边）

输出指标（Fusion.md 推荐）：
- 度分布（in/out/total）& 长尾程度（Gini）
- 连通分量大小分布、最大分量占比（按无向化连通分量）
- 重复边比例（同一 payer→payee 多次）
- reciprocity（互转比例）

In [ ]:
import networkx as nx


In [ ]:
def _gini(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    x = np.sort(x)
    if np.all(x == 0):
        return 0.0
    n = x.size
    cumx = np.cumsum(x)
    # Gini = (n+1 - 2*sum(cumx)/cumx[-1]) / n
    return float((n + 1 - 2 * (cumx.sum() / cumx[-1])) / n)

def _build_digraph(edges: pd.DataFrame, src: str, dst: str) -> nx.DiGraph:
    g = nx.DiGraph()
    # 去NaN
    e = edges[[src, dst]].dropna()
    # 强转字符串避免混类型
    e[src] = e[src].astype(str)
    e[dst] = e[dst].astype(str)
    g.add_edges_from(e.itertuples(index=False, name=None))
    return g

def _graph_metrics(g: nx.DiGraph) -> dict:
    n = g.number_of_nodes()
    m = g.number_of_edges()

    in_deg = np.array([d for _, d in g.in_degree()], dtype=float)
    out_deg = np.array([d for _, d in g.out_degree()], dtype=float)
    tot_deg = in_deg + out_deg

    # 无向化连通分量
    ug = g.to_undirected(as_view=False)
    comps = [len(c) for c in nx.connected_components(ug)] if n > 0 else []
    comps = np.array(comps, dtype=float)
    largest_comp_ratio = (comps.max() / n) if (n > 0 and comps.size > 0) else np.nan

    # 重复边比例（基于原始边序列统计，不使用Graph去重）
    # 这里假设 g 的边已经去重，所以重复边要从原数据里算（另算函数）

    # reciprocity：nx.reciprocity 返回整体互惠率（有向边中成对出现的比例）
    try:
        rec = nx.reciprocity(g)
    except Exception:
        rec = np.nan

    return {
        "nodes": int(n),
        "edges_unique": int(m),
        "in_deg_gini": _gini(in_deg),
        "out_deg_gini": _gini(out_deg),
        "total_deg_gini": _gini(tot_deg),
        "components": int(comps.size),
        "largest_comp_ratio": float(largest_comp_ratio) if largest_comp_ratio == largest_comp_ratio else np.nan,
        "reciprocity": float(rec) if rec == rec else np.nan,
        "in_deg_p99": float(np.quantile(in_deg, 0.99)) if in_deg.size else np.nan,
        "out_deg_p99": float(np.quantile(out_deg, 0.99)) if out_deg.size else np.nan,
    }

def _duplicate_edge_ratio(edge_df: pd.DataFrame, src: str, dst: str) -> float:
    e = edge_df[[src, dst]].dropna().copy()
    if len(e) == 0:
        return np.nan
    e[src] = e[src].astype(str)
    e[dst] = e[dst].astype(str)
    total = len(e)
    unique = e.drop_duplicates().shape[0]
    return float(1 - unique / total)


In [ ]:
# PaySim 图
p_edge_df = paysim[["nameOrig", "nameDest"]].copy()
p_dup = _duplicate_edge_ratio(p_edge_df, "nameOrig", "nameDest")
p_g = _build_digraph(p_edge_df, "nameOrig", "nameDest")
p_metrics = _graph_metrics(p_g)
p_metrics["duplicate_edge_ratio"] = p_dup

# 无标签图
u_edge_df = unlabeled[[UNLABELED_COLS["src"], UNLABELED_COLS["dst"]]].copy()
u_dup = _duplicate_edge_ratio(u_edge_df, UNLABELED_COLS["src"], UNLABELED_COLS["dst"])
u_g = _build_digraph(u_edge_df, UNLABELED_COLS["src"], UNLABELED_COLS["dst"])
u_metrics = _graph_metrics(u_g)
u_metrics["duplicate_edge_ratio"] = u_dup

graph_summary = pd.DataFrame([
    {"dataset": "PaySim", **p_metrics},
    {"dataset": "Unlabeled", **u_metrics},
])
graph_summary

In [ ]:
# 度分布可视化（log-log更容易看长尾）
def _plot_degree_ccdf(g: nx.DiGraph, title: str):
    deg = np.array([d for _, d in g.degree()], dtype=float)
    deg = deg[deg > 0]
    if deg.size == 0:
        print(f"{title}: 无有效度数")
        return
    xs = np.sort(deg)
    ccdf = 1.0 - np.arange(1, xs.size + 1) / xs.size
    plt.figure(figsize=(6,4))
    plt.plot(xs, ccdf, marker='.', linestyle='none')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(f"Degree CCDF (log-log): {title}")
    plt.xlabel("degree")
    plt.ylabel("P(Degree>=x)")
    plt.tight_layout()
    plt.show()

_plot_degree_ccdf(p_g, "PaySim")
_plot_degree_ccdf(u_g, "Unlabeled")


In [ ]:
# 连通分量大小分布（无向化）
def _component_sizes(g: nx.DiGraph) -> np.ndarray:
    ug = g.to_undirected(as_view=False)
    comps = [len(c) for c in nx.connected_components(ug)] if g.number_of_nodes() else []
    return np.array(comps, dtype=float)

p_cs = _component_sizes(p_g)
u_cs = _component_sizes(u_g)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
if p_cs.size:
    sns.histplot(np.log1p(p_cs), bins=100, stat="density", kde=True, ax=ax, color="#4c78a8", label="PaySim")
if u_cs.size:
    sns.histplot(np.log1p(u_cs), bins=100, stat="density", kde=True, ax=ax, color="#f58518", label="Unlabeled")
ax.set_title("连通分量大小分布（log1p空间）")
ax.set_xlabel("log1p(component_size)")
ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame({
    "dataset": ["PaySim", "Unlabeled"],
    "components": [int(p_cs.size), int(u_cs.size)],
    "median_comp": [float(np.median(p_cs)) if p_cs.size else np.nan, float(np.median(u_cs)) if u_cs.size else np.nan],
    "p99_comp": [float(np.quantile(p_cs, 0.99)) if p_cs.size else np.nan, float(np.quantile(u_cs, 0.99)) if u_cs.size else np.nan],
    "largest_comp_ratio": [p_metrics.get("largest_comp_ratio"), u_metrics.get("largest_comp_ratio")],
})

## 5) 汇总表 & 解读建议

这里把三层检验的核心数字汇总到一张表，便于写报告：
- 金额：KS/Wasserstein（raw/log1p）、尾部占比
- 节奏：log1p(间隔) 的 KS/Wasserstein
- 图结构：Gini、连通分量、重复边、reciprocity

解读建议（经验）：
- **图结构层**最能影响 GraphMAE 迁移性；若两边“最大连通分量占比、度长尾形态、重复边比例”差异很大，迁移证据会变弱。
- **节奏层**看形态即可：是否同样重尾/爆发式；不要强求时间尺度一致。
- **金额层**建议重点看 log1p 后的 KS/Wasserstein 与尾部占比（Top 1%/0.1%）。

In [ ]:
# 汇总表
summary_rows = []

# 金额
summary_rows.append({
    "section": "table.amount",
    "metric": "ks_stat_log1p",
    "paysim": np.nan,
    "unlabeled": np.nan,
    "value": float(dist_tbl.loc[dist_tbl["space"]=="log1p","ks_stat"].iloc[0]),
    "note": "log1p(amount) 的 KS statistic，越小越相似",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "wasserstein_log1p",
    "paysim": np.nan,
    "unlabeled": np.nan,
    "value": float(dist_tbl.loc[dist_tbl["space"]=="log1p","wasserstein"].iloc[0]),
    "note": "log1p(amount) 的 Wasserstein 距离，越小越相似",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "top1%_share",
    "paysim": float(tail_tbl.loc[tail_tbl["dataset"]=="PaySim","top1%_share"].iloc[0]),
    "unlabeled": float(tail_tbl.loc[tail_tbl["dataset"]=="Unlabeled","top1%_share"].iloc[0]),
    "value": np.nan,
    "note": "Top 1% 金额贡献占比，越接近越好（重尾程度）",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "top0.1%_share",
    "paysim": float(tail_tbl.loc[tail_tbl["dataset"]=="PaySim","top0.1%_share"].iloc[0]),
    "unlabeled": float(tail_tbl.loc[tail_tbl["dataset"]=="Unlabeled","top0.1%_share"].iloc[0]),
    "value": np.nan,
    "note": "Top 0.1% 金额贡献占比，越接近越好",
})

# 节奏
summary_rows.append({
    "section": "timing.interarrival",
    "metric": "ks_stat_log1p_delta",
    "paysim": np.nan,
    "unlabeled": np.nan,
    "value": np.nan if ks_dt is None else float(ks_dt.statistic),
    "note": "log1p(间隔) KS statistic，越小越相似（形态）",
})
summary_rows.append({
    "section": "timing.interarrival",
    "metric": "wasserstein_log1p_delta",
    "paysim": np.nan,
    "unlabeled": np.nan,
    "value": float(wd_dt) if wd_dt == wd_dt else np.nan,
    "note": "log1p(间隔) Wasserstein 距离，越小越相似",
})

# 图结构（两边各自值）
for k in [
    "nodes","edges_unique","components","largest_comp_ratio",
    "duplicate_edge_ratio","reciprocity","total_deg_gini","in_deg_gini","out_deg_gini"
 ]:
    summary_rows.append({
        "section": "graph.structure",
        "metric": k,
        "paysim": p_metrics.get(k) if k in p_metrics else (p_dup if k=="duplicate_edge_ratio" else np.nan),
        "unlabeled": u_metrics.get(k) if k in u_metrics else (u_dup if k=="duplicate_edge_ratio" else np.nan),
        "value": np.nan,
        "note": "结构统计（对齐关注差异幅度）",
    })

summary = pd.DataFrame(summary_rows)

# ========== 新增：relative_diff（相对差异）==========
def _rel_diff(p, u):
    """计算相对差异：|p - u| / max(|p|, |u|, 1e-9)，返回百分比"""
    if pd.isna(p) or pd.isna(u):
        return np.nan
    denom = max(abs(p), abs(u), 1e-9)
    return abs(p - u) / denom

summary["relative_diff"] = summary.apply(
    lambda r: _rel_diff(r["paysim"], r["unlabeled"]) if pd.notna(r["paysim"]) and pd.notna(r["unlabeled"]) else np.nan,
    axis=1
)

# ========== 新增：norm_wasserstein（归一化 Wasserstein）==========
# 对金额和间隔的 Wasserstein 除以 PaySim 的 IQR（log1p 空间），使其变成无量纲
pa_log_iqr = np.subtract(*np.percentile(np.log1p(pa), [75, 25])) if len(pa) else 1.0
p_log_dt_iqr = np.subtract(*np.percentile(p_log, [75, 25])) if len(p_log) else 1.0

def _norm_wd(metric, raw_wd):
    if pd.isna(raw_wd):
        return np.nan
    if metric == "wasserstein_log1p":
        return raw_wd / pa_log_iqr if pa_log_iqr > 0 else np.nan
    elif metric == "wasserstein_log1p_delta":
        return raw_wd / p_log_dt_iqr if p_log_dt_iqr > 0 else np.nan
    return np.nan

summary["norm_wasserstein"] = summary.apply(
    lambda r: _norm_wd(r["metric"], r["value"]),
    axis=1
)

summary

## 📌 summary 表指标解读（含公式 & “多相似算相似”）

> 结论先说：这些指标没有统一的官方“合格线”。更推荐用**相对比较**：PaySim vs 目标域是否“同量级 + 同形态 + 同趋势”。下面给出常用解释与经验阈值（可写进报告）。

---

### A. 表格层：金额分布（`table.amount.*`）

#### 1) `ks_stat_log1p`（Kolmogorov–Smirnov 统计量）
- **含义**：比较两组样本的经验CDF（累积分布）最大差距。
- **公式**：设两域的经验分布函数为 $F_n(x), G_m(x)$：
  $$ D_{KS} = \sup_x \left|F_n(x)-G_m(x)\right| $$
- **范围**：$[0,1]$，越小越相似。
- **为什么用 log1p**：金额通常重尾，直接在 raw 空间 KS 常被尾部极值“主导”；用 $\log(1+x)$ 能更稳地看 **形态**。
- **经验解读（log1p 空间）**：
  - $D_{KS} \le 0.10$：分布形态通常算“很接近”
  - $0.10 < D_{KS} \le 0.20$：中等差异（仍可能可迁移，但要结合图结构层）
  - $D_{KS} > 0.20$：差异偏大（迁移证据会变弱）
- **注意**：样本量很大时 p-value 往往非常小，不要只看 `ks_pvalue`，主要看 `ks_stat`。

#### 2) `wasserstein_log1p`（Wasserstein-1 / Earth Mover’s Distance）
- **含义**：把一个分布“搬运”成另一个分布所需的最小平均代价（更直观地衡量差别）。
- **1D 形式（直觉/常用定义）**：
  $$ W_1(P,Q)=\int_{-\infty}^{\infty} \left|F(x)-G(x)\right|\,dx $$
  在 1D 上也可理解为排序后点对点距离的平均量级。
- **范围**：$[0,\infty)$，越小越相似。
- **经验解读**：这个值**依赖单位与尺度**（即使在 log1p 空间也如此），建议：
  - 在报告里用“PaySim vs Unlabeled 的 Wasserstein 值”作为一个数字，并配合 KS+分位数曲线解释；
  - 或进一步做归一化（例如除以 PaySim 的 $IQR$），得到更可比的“无量纲距离”（如需我可以再加一列）。

#### 3) `top1%_share` / `top0.1%_share`（尾部贡献占比）
- **含义**：衡量分布重尾程度：Top 1%（或 Top 0.1%）金额样本贡献了总金额的多少。
- **公式**：设阈值 $t_q$ 为 $q$ 分位数（如 $q=0.99$）：
  $$ \text{TailShare}(q)=\frac{\sum_{i: x_i\ge t_q} x_i}{\sum_i x_i} $$
- **范围**：$[0,1]$，越大表示越“重尾”。
- **经验解读**：两域的尾部占比如果**相差不超过 20%~30%（相对差）**，通常可认为尾部形态相近。

---

### B. 时序层：交易间隔（`timing.interarrival.*`）

#### 4) `ks_stat_log1p_delta` / `wasserstein_log1p_delta`
- **对齐对象**：
  - PaySim：同一账户相邻交易的 $\Delta step$
  - 无标签：同一账户相邻交易的 $\Delta t$（秒）
- **关键点**：两边单位不同，因此我们只希望它们的 **分布形态**近似（重尾、爆发性、长间隔占比等），而不是数值完全一致。
- **经验阈值（log1p 空间）**：可沿用金额的 KS 经验档位：
  - $D_{KS} \le 0.10$：节奏形态很像
  - $0.10-0.20$：存在差异，需要结合图结构判断
  - $>0.20$：差异偏大，说明业务节奏机制可能不同

---

### C. 图结构层（最重要） ：账户交易图（`graph.structure.*`）

#### 5) `nodes` / `edges_unique` / `components`
- `nodes`：图中账户节点数（唯一账户数）。
- `edges_unique`：唯一有向边数（唯一 payer→payee 关系）。
- `components`：无向化后连通分量个数。分量越多、越碎，GNN 的消息传播越受限。
- **经验解读**：两域如果 `components/nodes`（碎片化率）差异很大，GraphMAE 迁移性通常会变差。

#### 6) `largest_comp_ratio`（最大连通分量占比）
- **含义**：最大连通分量节点数 / 总节点数。
- **公式**：设最大连通分量大小为 $|C_{max}|$，总节点数为 $N$：
  $$ r=\frac{|C_{max}|}{N} $$
- **范围**：$[0,1]$，越大表示图越“连”。
- **经验解读**：
  - 两域的 $r$ 若都很小（比如都 < 0.05）说明都很碎：这本身不利于GNN，但“相似性”可能还可以。
  - 若一边很大（如 0.3）另一边很小（如 0.01），结构机制差异很大：迁移证据明显变弱。

#### 7) `duplicate_edge_ratio`（重复边比例）
- **含义**：同一 payer→payee 关系重复出现的程度（交易关系是否重复）。
- **公式**：设总交易边数 $M$，去重后唯一边数 $M_{uniq}$：
  $$ \text{DupRatio}=1-\frac{M_{uniq}}{M} $$
- **范围**：$[0,1]$，越大表示重复交易关系越多。
- **经验解读**：这是判断“结构是否可挖”的关键指标之一：
  - 两域都很低（接近 0）：说明关系高度一次性，图结构信号弱，GNN 更依赖节点/边特征。
  - 两域都中等或都高：说明都存在重复交易关系，更有利于图方法迁移。
  - 一高一低：迁移风险增大。

#### 8) `reciprocity`（互惠率/互转比例）
- **含义**：有向图中，边 $(u\to v)$ 也存在 $(v\to u)$ 的比例。
- **直觉**：互转越多，图更像“往来账户网络”；互转很少可能更像“单向支付链”。
- **范围**：$[0,1]$（NetworkX 的定义是整体互惠率），越大互转越多。
- **经验解读**：两域若相差>2倍（例如 0.02 vs 0.10），说明交互模式差异较明显。

#### 9) `total_deg_gini` / `in_deg_gini` / `out_deg_gini`（度分布不均衡程度）
- **含义**：衡量度分布是否“长尾/头部集中”。
- **Gini 公式（离散样本版）**：对非负序列 $x$ 升序排序 $x_{(i)}$，样本量 $n$，累积和 $S=\sum_i x_i$：
  $$ G=\frac{n+1-2\cdot\frac{\sum_{i=1}^{n}\sum_{j=1}^{i} x_{(j)}}{S}}{n} $$
  （Notebook里用的是等价写法）
- **范围**：$[0,1]$，越大说明“少数节点占据大量连接”（更长尾）。
- **经验解读**：如果两域 Gini 差异很大（比如 0.2 vs 0.7），说明网络中心化程度不同，迁移证据变弱。

---

## ✅ 怎么定“相似”的实用准则（建议写报告的口径）

建议用“三层证据 + 一句结论”而不是单阈值：
1. **金额层（log1p）**：`ks_stat_log1p` ≤ 0.1~0.2 且尾部占比差异不大（≤20%~30%相对差）。
2. **节奏层（log1p）**：`ks_stat_log1p_delta` ≤ 0.1~0.2；形态图看起来同为重尾/爆发。
3. **图结构层（核心）**：`largest_comp_ratio`、`duplicate_edge_ratio`、`total_deg_gini` 的量级接近（比如都在同一档位：低/中/高），且 CCDF 形态相近。

最后一句可写成：
- "三层指标整体处于同量级，图结构形态接近，因此可以将 PaySim 上的 model selection 作为无标签域的弱证据"；或
- "图结构差异显著（碎图程度/重复边比例/度长尾差很多），因此 PaySim 迁移证据有限，需要更多 label-free 证据（稳定性、弱规则命中、抽检）"。

---

## 📊 新增列解释：`relative_diff` 与 `norm_wasserstein`

为了让 `summary` 表更直观地展示两域差异程度，本 Notebook 新增了以下两列：

### 10) `relative_diff`（相对差异）
- **含义**：当某指标在两域各有一个值（`paysim` 列 与 `unlabeled` 列）时，计算它们之间的 **相对差异**，便于跨量级比较。
- **公式**：
  $$ \text{RelDiff} = \frac{|p - u|}{\max(|p|,|u|,\epsilon)} $$
  其中 $\epsilon=10^{-9}$ 用于防止除零。
- **范围**：$[0, 1]$（如果两值同号且一个为 0 时可能接近 1；如果完全相等则为 0）。
- **经验解读**：
  - $\le 0.20$（20%）：差异较小，可认为"量级相近"
  - $0.20 \sim 0.50$：中等差异，需结合其他指标判断
  - $> 0.50$（50%）：差异较大，迁移证据变弱
- **适用指标**：`top1%_share`、`top0.1%_share`、`largest_comp_ratio`、`duplicate_edge_ratio`、`reciprocity`、`*_gini` 等同时有 `paysim` 和 `unlabeled` 值的行。

### 11) `norm_wasserstein`（归一化 Wasserstein）
- **含义**：原始 Wasserstein 距离依赖分布尺度，不同场景数值差很大。用 PaySim 在 log1p 空间的 **IQR（四分位距）** 做分母，将其转成 **无量纲** 的"相对距离"。
- **公式**：
  $$ \text{NormWD} = \frac{W_1^{(\log1p)}}{\mathrm{IQR}_{PaySim}^{(\log1p)}} $$
  其中 $\mathrm{IQR} = Q_{75} - Q_{25}$（log1p 变换后的 PaySim 数据）。
- **范围**：$[0,\infty)$，理论上可大于 1（若距离超过一个 IQR）。
- **经验解读**：
  - $\le 0.5$：分布在 log1p 空间接近（不到半个 IQR 的距离）
  - $0.5 \sim 1.0$：中等差异
  - $> 1.0$：差异较大（超过 PaySim 本身的中间 50% 波动范围）
- **适用指标**：`wasserstein_log1p`（金额）、`wasserstein_log1p_delta`（间隔）。

> **使用建议**：在写报告时，可直接引用 `relative_diff` 列来说明"两域的 xxx 指标相对差异为 xx%"，或引用 `norm_wasserstein` 来说明"金额分布的归一化 Wasserstein 仅为 0.3 IQR，说明形态非常接近"。